In [2]:
import torch
import math
import numpy as np

In [3]:
# Inputs
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

#### 1. Forward pass

In [4]:
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability

In [6]:
z

tensor(6.7000)

In [7]:
y_pred

tensor(0.9988)

In [5]:
# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

In [8]:
# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [9]:
loss

tensor(6.7012)

### 2. Backwade pass

In [10]:
# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [11]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


### Autograd

In [15]:
# Inputs
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0,requires_grad=True)  # Weight
b = torch.tensor(0.0,requires_grad=True)  # Bias

In [16]:
# forward pass
z = w*x + b
y_pred = torch.sigmoid(z)  # Predicted probability

In [17]:
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [18]:
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [19]:
loss = binary_cross_entropy_loss(y_pred, y)

In [20]:
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [21]:
loss.backward()

In [22]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


### Clearing Grad

In [24]:
x1 = torch.tensor(2.0,requires_grad=True)

In [25]:
x1

tensor(2., requires_grad=True)

In [37]:
y = x1 ** 2

In [38]:
y.backward()

In [39]:
x1.grad

tensor(4.)

In [36]:
x1.grad.zero_()

tensor(0.)

### disable gradient tracking

##### option 1 - requires_grad_(False)

In [49]:
x2 = torch.tensor(4.0,requires_grad=True)
y = x2 ** 2

In [50]:
y.backward()

In [51]:
x2.grad

tensor(8.)

In [52]:
x2

tensor(4., requires_grad=True)

In [53]:
x2.requires_grad_(False)

tensor(4.)

In [54]:
y = x2 ** 2
y

tensor(16.)

In [47]:
y.backward()

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

##### option 2 - detach()

In [61]:
x3 = torch.tensor(4.0, requires_grad=True)

In [62]:
x3

tensor(4., requires_grad=True)

In [63]:
z = x3.detach()
z

tensor(4.)

In [64]:
y = x3 ** 2
y

tensor(16., grad_fn=<PowBackward0>)

In [65]:
y1 = z ** 2
y1

tensor(16.)

##### option 3 - torch.no_grad()

In [66]:
x4 = torch.tensor(4.0,requires_grad=True)
x5 = torch.tensor(5.0,requires_grad=True)

In [76]:
with torch.no_grad():
    y100 = x4 ** 4
y200 = x4 ** 4

In [77]:
y100 , y200

(tensor(256.), tensor(256., grad_fn=<PowBackward0>))